# 02 — DCF Valuation, Sensitivity & Scenarios

Covers `src/valuation/wacc.py`, `dcf.py`, and `sensitivity.py`.

- **FCFF** = `EBIT*(1-Tc) + D&A - CapEx - ΔNWC`
- **TV (Gordon)**: `FCFF_{n+1}/(WACC-g)` — requires `WACC > g` (validated)
- **TV (Exit)**: `EBITDA_n × Multiple`
- Sensitivity grids and Bear/Base/Bull scenarios reuse the same DCF core.

In [ ]:
from src.financial_model.assumptions import ModelAssumptions, BEAR_SCENARIO, BASE_SCENARIO, BULL_SCENARIO
from src.financial_model.three_statement import ThreeStatementModel
from src.valuation.wacc import calculate_cost_of_equity, calculate_wacc
from src.valuation.dcf import DCFModel
from src.valuation.sensitivity import sensitivity_wacc_growth, sensitivity_wacc_exit_multiple, scenario_comparison
import pandas as pd

assumptions = ModelAssumptions(
    historical_revenue=500, revenue_growth=[0.06,0.05,0.04,0.03,0.03],
    gross_margin=0.40, opex_margin=0.20, da_pct_revenue=0.03, capex_pct_revenue=0.04,
    tax_rate=0.25, ar_days=45, inventory_days=30, ap_days=40,
    beginning_cash=80, beginning_ar=62, beginning_inventory=25, beginning_ppe=200,
    beginning_other_assets=20, beginning_ap=30, beginning_debt=150, beginning_other_liabilities=10,
    beginning_equity=197, beginning_retained_earnings=100,
    interest_rate=0.05, dividends_pct_net_income=0.20, share_count=10, forecast_years=5,
)
model = ThreeStatementModel(assumptions)
model.income_statement.round(1)


## 1. WACC via CAPM
`Ke = Rf + β·ERP`, `Kd_after = Kd*(1-Tc)`, `WACC = (E/V)Ke + (D/V)Kd_after`.

In [ ]:
ke = calculate_cost_of_equity(risk_free_rate=0.03, beta=1.1, equity_risk_premium=0.05)
wacc = calculate_wacc(market_cap=800, debt=150, cost_of_equity=ke, cost_of_debt=0.05, tax_rate=0.25)
print(f"Ke={ke:.2%}, WACC={wacc:.2%}")
# Error handling demo:
try:
    calculate_wacc(0,0,0.08,0.05,0.25)
except ValueError as e:
    print("Caught expected:", e)


## 2. Build DCF (both methods)
`DCFModel.from_three_statement` extracts Revenue/EBIT/D&A/CapEx/ΔWC and uses beginning cash/debt for `Equity = EV + Cash - Debt`.

In [ ]:
dcf = DCFModel.from_three_statement(model, wacc=wacc, terminal_growth=0.02, exit_multiple=10)
display(dcf.build().round(1))
print(dcf.summary("growth"))
print(dcf.summary("multiple"))


In [ ]:
from src.visualization.charts import plot_fcff_forecast
plot_fcff_forecast(dcf.to_dataframe(), save_path="../outputs/fcff_forecast.png")


## 3. Sensitivity: WACC × Growth and WACC × Multiple
Each cell is a full DCF repriced at that pair; `WACC ≤ g` → `NaN` (invalid Gordon).

In [ ]:
wacc_range = [0.07,0.08,0.09,0.10,0.11]
growth_range = [0.01,0.015,0.02,0.025,0.03]
sens_g = sensitivity_wacc_growth(dcf.inputs, wacc_range, growth_range)
sens_g.round(1)


In [ ]:
from src.visualization.charts import plot_sensitivity_heatmap
plot_sensitivity_heatmap(sens_g, title="WACC × Terminal Growth → Implied Price", save_path="../outputs/dcf_sensitivity.png", base_wacc="9.0%", base_growth="2.0%")
mult_range = [8,9,10,11,12]
sens_m = sensitivity_wacc_exit_multiple(dcf.inputs, wacc_range, mult_range)
display(sens_m.round(1))
plot_sensitivity_heatmap(sens_m, title="WACC × Exit Multiple → Implied Price", save_path="../outputs/dcf_sensitivity_multiple.png")


## 4. Bear / Base / Bull scenarios
Each scenario overrides growth/margin/CapEx and optionally WACC/g. Comparison shows EV/Equity/Price/Upside.

In [ ]:
scen = scenario_comparison(assumptions, scenarios=[BEAR_SCENARIO,BASE_SCENARIO,BULL_SCENARIO],
    wacc_map={"Bear":0.10,"Base":0.09,"Bull":0.08}, terminal_growth_map={"Bear":0.01,"Base":0.02,"Bull":0.025}, current_price=80)
scen.round(2)
